# 评估器核对（打分前跑一次）

预锁定第 5 节规定：任何一站打分之前，先在 merged_v5 的 val 上，用 v0 比一下**本仓库的评估器**（`dl.py score` 用的那一套）和
**Ultralytics 8.4 的 val**。11 类 mAP50 差距在 0.01 以内才算通过。

跑完把生成的 `evaluator_check.json` 放进仓库的 `results/`，commit。CPU 就够，大概 10 分钟。

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mixed_dataset"     # 放 merged_v5.zip 和 v0 ONNX 的文件夹
REPO_URL = "https://github.com/xiaolin200206/Snapdurian_deploy.git"
ULTRALYTICS = "8.4.182"
ONNX_NAME = "durian_11cls_640_v5.onnx"

In [ ]:
import os, sys, json, subprocess, shutil, zipfile, hashlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("不在 Colab 里")
DRIVE = Path(DRIVE_DIR)
REPO = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}", "onnxruntime", "pyyaml", "onnx", "onnxslim"], check=True)
sys.path.insert(0, str(REPO)); os.chdir(REPO)
PRE = json.loads((REPO / "sequential" / "seq_prelock.json").read_text(encoding="utf-8"))
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()
def unzip_once(zp, dest, marker):
    if (dest / marker).exists():
        print("已解压：", dest / marker); return
    print("解压", zp.name, "→", dest)
    local = Path("/content") / zp.name
    if not local.exists(): shutil.copy2(zp, local)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local) as z: z.extractall(dest)
unzip_once(DRIVE / "merged_v5.zip", REPO / "data", "merged_v5")
MERGED = REPO / "data" / "merged_v5"
print("merged_v5 val 图片：", len(list((MERGED / "images" / "val").glob("*.*"))))

In [ ]:
onnx_path = DRIVE / ONNX_NAME
assert sha256(onnx_path) == PRE["v0"]["onnx"]["sha256"], "这个 ONNX 不是预锁定里的 v0！"
print("v0 哈希一致")
import yaml
from dlcore.common import PRODUCT
vy = Path("/content/v0_val.yaml")
vy.write_text(yaml.safe_dump({"train": str(MERGED / "images" / "val"), "val": str(MERGED / "images" / "val"),
                              "nc": 11, "names": PRODUCT}, sort_keys=False))
from ultralytics import YOLO
r = YOLO(str(onnx_path), task="detect").val(data=str(vy), split="val", imgsz=640, batch=1, conf=0.001, iou=0.7,
                                            device="cpu", plots=False, verbose=False)
ul = {r.names[int(c)]: float(r.box.ap50[i]) for i, c in enumerate(r.box.ap_class_index)}
ul_map = float(r.box.map50)
print(f"Ultralytics mAP50 = {ul_map:.4f}")

In [ ]:
from dlcore.evalset import evaluate_folder
ours = evaluate_folder(str(onnx_path), MERGED / "images" / "val", MERGED / "labels" / "val")
print(f"本仓库 mAP50 = {ours['mAP50']:.4f}")
ref = PRE["v0"]["reference_val_onnx"]
rows = []
for c in PRODUCT:
    a, b, rr = ul.get(c), ours["AP50"].get(c), ref["per_class"].get(c, {}).get("ap50")
    rows.append((c, a, b, rr))
    f4 = lambda x: "  n/a " if x is None else f"{x:.4f}"
    print(f"  {c:<20} Ultralytics {f4(a)}   本仓库 {f4(b)}   差 {f4(None if a is None or b is None else b - a)}   （预锁定参考 {f4(rr)}）")
d_map = ours["mAP50"] - ul_map
max_cls = max(abs(b - a) for _, a, b, _ in rows if a is not None and b is not None)
ok = abs(d_map) <= 0.01
print(f"\nmAP50 差 {d_map:+.4f}，单类最大差 {max_cls:.4f} → {'✓ 通过' if ok else '✗ 没通过：先别打分，把结果发给我'}")
out = {"ultralytics_version": ULTRALYTICS, "v0_sha256": PRE["v0"]["onnx"]["sha256"], "n_images": ours["n_images"],
       "ultralytics_mAP50": ul_map, "ours_mAP50": ours["mAP50"], "diff": d_map, "max_class_diff": max_cls,
       "tolerance": 0.01, "passed": ok, "per_class": {c: {"ultralytics": a, "ours": b} for c, a, b, _ in rows}}
(DRIVE / "evaluator_check.json").write_text(json.dumps(out, indent=2))
print("已存到 Drive：", DRIVE / "evaluator_check.json", "→ 放进仓库 results/ 并 commit")